# 03 — Train DeBERTa (baseline: human vs AI)

This notebook **fine-tunes** `microsoft/deberta-v3-base` on **one** LOGO rotation using the usual classification loss (cross-entropy). It uses your CSVs from `02_build_splits.ipynb`.

**Before you run:** On Colab, run `00_setup.ipynb` once so the repo lives at `MyDrive/ECS111FinalProject` with `data/splits/...`.

**Pick** `ROTATION` in `0..3`. Re-run for each rotation.

**Saves to:** `checkpoints/deberta_ce/rotation_{ROTATION}/best_hf/` (folder is gitignored — keep it on Drive in Colab).

**Local Windows GPU:** prefer `python scripts/train_deberta.py --rotation 0` from PowerShell (more stable than Jupyter). This notebook is synced for the same settings (fp32, `processing_class`, small batches).

In [1]:
# Installs — run once per new Colab runtime
%pip install -q "transformers>=4.36" "datasets>=2.16" "accelerate>=0.26" "evaluate>=0.4" sentencepiece protobuf scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: C:\Users\soula\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## 1) Where the project lives

- **Colab:** this notebook mounts Drive and uses `MyDrive/ECS111FinalProject`.
- **Local Jupyter:** set environment variable `ECS111_PROJECT_DIR` to your clone root (folder that contains `data/`). Otherwise it assumes the parent of the current working directory.

In [2]:
import os
import random

import numpy as np
import torch
from datasets import load_dataset
from sklearn.metrics import f1_score, roc_auc_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

# --- Colab: mount and use Drive path ---
if os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/ECS111FinalProject"
else:
    PROJECT_DIR = os.environ.get("ECS111_PROJECT_DIR")
    if not PROJECT_DIR:
        cwd = os.getcwd()
        if os.path.isdir(os.path.join(cwd, "data", "splits")):
            PROJECT_DIR = cwd
        elif os.path.isdir(os.path.join(os.path.dirname(cwd), "data", "splits")):
            PROJECT_DIR = os.path.dirname(cwd)
        else:
            PROJECT_DIR = cwd

# LOGO rotation 0..3 (must match folders under data/splits/)
ROTATION = 0

MODEL_NAME = "microsoft/deberta-v3-base"
EPOCHS = 3
LR = 2e-5
SEED = 42

# Local PC (e.g. RTX 4060 8GB): fp32 + smaller batches (bf16 caused loss=0 / nan grads)
ON_COLAB = os.path.isdir("/content/drive/MyDrive")
LOCAL_GPU = (not ON_COLAB) and torch.cuda.is_available()
USE_BF16 = (not LOCAL_GPU) and torch.cuda.is_available() and torch.cuda.is_bf16_supported()
if LOCAL_GPU:
    MAX_LENGTH = 256
    BATCH_SIZE = 4
    GRAD_ACCUM = 4
else:
    MAX_LENGTH = 384
    BATCH_SIZE = 16
    GRAD_ACCUM = 1

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

os.chdir(PROJECT_DIR)
print("PROJECT_DIR:", PROJECT_DIR)
print("ROTATION:", ROTATION)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print(f"train: batch={BATCH_SIZE}, accum={GRAD_ACCUM}, max_len={MAX_LENGTH}, bf16={USE_BF16}, fp16=False")

PROJECT_DIR: C:\Users\soula\Downloads\ECS111 final proj\ECS111 final proj\ECS111FinalProject
ROTATION: 0
CUDA: True
GPU: NVIDIA GeForce RTX 4060
train: batch=4, accum=4, max_len=256, bf16=True, fp16=False


In [3]:
train_path = os.path.join(PROJECT_DIR, "data", "splits", f"rotation_{ROTATION}", "train.csv")
val_path = os.path.join(PROJECT_DIR, "data", "splits", f"rotation_{ROTATION}", "val_indist.csv")

assert os.path.isfile(train_path), f"Missing {train_path} — run 02_build_splits first"
assert os.path.isfile(val_path), f"Missing {val_path}"

out_dir = os.path.join(PROJECT_DIR, "checkpoints", "deberta_ce", f"rotation_{ROTATION}")
os.makedirs(out_dir, exist_ok=True)
print("train:", train_path)
print("val:  ", val_path)
print("out:  ", out_dir)

train: C:\Users\soula\Downloads\ECS111 final proj\ECS111 final proj\ECS111FinalProject\data\splits\rotation_0\train.csv
val:   C:\Users\soula\Downloads\ECS111 final proj\ECS111 final proj\ECS111FinalProject\data\splits\rotation_0\val_indist.csv
out:   C:\Users\soula\Downloads\ECS111 final proj\ECS111 final proj\ECS111FinalProject\checkpoints\deberta_ce\rotation_0


In [4]:
raw = load_dataset("csv", data_files={"train": train_path, "validation": val_path})
print(raw)

DatasetDict({
    train: Dataset({
        features: ['review_id', 'category', 'parent_asin', 'rating', 'title', 'text', 'timestamp', 'label', 'generator'],
        num_rows: 11994
    })
    validation: Dataset({
        features: ['review_id', 'category', 'parent_asin', 'rating', 'title', 'text', 'timestamp', 'label', 'generator'],
        num_rows: 1496
    })
})


In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)


def safe_str(x):
    if x is None:
        return ""
    s = str(x).strip()
    if s.lower() in ("nan", "none", ""):
        return ""
    return s


def prep(batch):
    titles = batch["title"]
    texts = batch["text"]
    labels = batch["label"]
    pieces = []
    for t, te in zip(titles, texts):
        piece = (safe_str(t) + " " + safe_str(te)).strip()
        pieces.append(piece if piece else safe_str(te))
    return {"text": pieces, "labels": labels}


def tokenize(batch):
    enc = tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )
    enc["labels"] = [int(x) for x in batch["labels"]]
    return enc


drop_cols = raw["train"].column_names
ds = raw.map(prep, batched=True, remove_columns=drop_cols)
ds = ds.map(tokenize, batched=True, remove_columns=["text"])
print("example keys:", ds["train"][0].keys())

example keys: dict_keys(['labels', 'input_ids', 'token_type_ids', 'attention_mask'])


In [6]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "human", 1: "ai"},
    label2id={"human": 0, "ai": 1},
    problem_type="single_label_classification",
    dtype=torch.float32,
)
if torch.cuda.is_available():
    model = model.cuda()

collator = DataCollatorWithPadding(tokenizer=tokenizer)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    probs = torch.softmax(torch.tensor(logits, dtype=torch.float32), dim=-1).numpy()[:, 1]
    out = {"f1_ai": f1_score(labels, preds, pos_label=1)}
    try:
        out["roc_auc"] = float(roc_auc_score(labels, probs))
    except ValueError:
        out["roc_auc"] = float("nan")
    return out


args = TrainingArguments(
    output_dir=out_dir,
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    warmup_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_ai",
    greater_is_better=True,
    save_total_limit=2,
    seed=SEED,
    report_to="none",
    bf16=USE_BF16,
    fp16=False,
    dataloader_num_workers=0,
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    processing_class=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer.train()
save_path = os.path.join(out_dir, "best_hf")
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print("Saved:", save_path)

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.den

Epoch,Training Loss,Validation Loss,F1 Ai,Roc Auc
1,0.000000,nan,0.000000,nan
2,0.000000,nan,0.000000,nan
3,0.000000,nan,0.000000,nan


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved: C:\Users\soula\Downloads\ECS111 final proj\ECS111 final proj\ECS111FinalProject\checkpoints\deberta_ce\rotation_0\best_hf


## Evaluate this rotation + summary row

Scores `test_indist` and `test_crossgen`, then prints one row in the **same table format** as the TF-IDF baseline. After all 4 rotations are trained, run **`04_eval_deberta_summary.ipynb`** for the full table + `runs/deberta_ce_summary.csv`.

In [2]:
import sys

sys.path.insert(0, os.path.join(PROJECT_DIR, "src"))
from eval_summary import make_rotation_row, print_summary, score_binary

SUMMARY_TITLE = "--- DEBERTA + CE BASELINE SUMMARY ---"


def load_eval_csv(split_name):
    path = os.path.join(
        PROJECT_DIR, "data", "splits", f"rotation_{ROTATION}", f"{split_name}.csv"
    )
    raw = load_dataset("csv", data_files={"test": path})["test"]
    drop = raw.column_names
    ds = raw.map(prep, batched=True, remove_columns=drop)
    return ds.map(tokenize, batched=True, remove_columns=["text"])


def predict_labels_probs(model, eval_ds):
    eval_trainer = Trainer(
        model=model,
        processing_class=tokenizer,
        data_collator=collator,
    )
    out = eval_trainer.predict(eval_ds)
    logits = out.predictions
    labels = out.label_ids
    preds = np.argmax(logits, axis=-1)
    probs = torch.softmax(torch.tensor(logits, dtype=torch.float32), dim=-1).numpy()[:, 1]
    return labels, preds, probs


model = AutoModelForSequenceClassification.from_pretrained(save_path, dtype=torch.float32)
if torch.cuda.is_available():
    model = model.cuda()
ds_indist = load_eval_csv("test_indist")
ds_cross = load_eval_csv("test_crossgen")

y_i, p_i, pr_i = predict_labels_probs(model, ds_indist)
y_c, p_c, pr_c = predict_labels_probs(model, ds_cross)

si = score_binary(y_i, p_i, pr_i)
sc = score_binary(y_c, p_c, pr_c)
row = make_rotation_row(ROTATION, si["f1"], si["auc"], sc["f1"], sc["auc"])

print_summary(SUMMARY_TITLE, [row])
print(f"rotation_{ROTATION} done. Train rotations 0-3, then run 04_eval_deberta_summary.ipynb for the full table.")

NameError: name 'os' is not defined